# The Standard news scraper

This notebook downloads English news articles from [The Standard](https://www.thestandard.com.hk/) for an inclusive date range.

The scraper writes two local outputs:

- Article text files with content only: `C:\Program Files\Studying\coding\RAG_project\data\the_standard_news`
- One metadata CSV file for the run: `C:\Program Files\Studying\coding\RAG_project\data\the_standard_news_metadata`

The main function is `scrape_standard_news(start_date, end_date)`. Use dates in `DD-MM-YYYY` format, for example `scrape_standard_news("01-05-2025", "01-05-2025")` for one day.

In [1]:
from __future__ import annotations

import csv
import html
import re
from datetime import date, datetime, timezone
from html.parser import HTMLParser
from pathlib import Path
from time import sleep
from typing import Any
from urllib.parse import urljoin
from zoneinfo import ZoneInfo

import requests

PROJECT_ROOT = Path(r"C:\Program Files\Studying\coding\RAG_project")
NEWS_DIR = PROJECT_ROOT / "data" / "the_standard_news"
METADATA_DIR = PROJECT_ROOT / "data" / "the_standard_news_metadata"

STANDARD_BASE_URL = "https://www.thestandard.com.hk"
STANDARD_CONTENT_PROXY_URL = f"{STANDARD_BASE_URL}/api/content"
HK_TIMEZONE = ZoneInfo("Asia/Hong_Kong")

# The function keeps exactly two arguments. Change this constant if you want more sections later.
STANDARD_CHANNEL_SLUGS = ("news",)


class StandardArticleTextExtractor(HTMLParser):
    """Extract body paragraphs from The Standard article HTML while skipping ads and page furniture."""

    def __init__(self) -> None:
        super().__init__(convert_charrefs=True)
        self._inside_article_body = False
        self._article_depth = 0
        self._skip_depth = 0
        self._parts: list[str] = []
        self._block_tags = {"p", "br", "li", "blockquote"}
        self._skip_tags = {"script", "style", "noscript", "iframe", "svg", "form", "button", "img", "picture", "source"}
        self._skip_class_keywords = {
            "ad-",
            "ad_",
            "ad-container",
            "advertisement",
            "interscroller",
            "taboola",
            "related",
            "share",
            "toolbar",
            "caption",
        }

    def _class_text(self, attrs: list[tuple[str, str | None]]) -> str:
        """Return lower-cased class/id text for simple container matching."""
        return " ".join(value or "" for name, value in attrs if name in {"class", "id", "role", "aria-label"}).lower()

    def _should_skip(self, tag: str, attrs: list[tuple[str, str | None]]) -> bool:
        """Return True for tags or containers that are not article content."""
        if tag in self._skip_tags:
            return True
        attr_text = self._class_text(attrs)
        return any(keyword in attr_text for keyword in self._skip_class_keywords)

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        """Start collecting only inside the article-detail text section."""
        attr_text = self._class_text(attrs)

        if not self._inside_article_body and "article-detail__text-section" in attr_text:
            self._inside_article_body = True
            self._article_depth = 1
            return

        if not self._inside_article_body:
            return

        self._article_depth += 1

        if self._skip_depth > 0:
            self._skip_depth += 1
            return

        if self._should_skip(tag, attrs):
            self._skip_depth = 1
            return

        if tag in self._block_tags:
            self._parts.append("\n")

    def handle_endtag(self, tag: str) -> None:
        """Stop collecting when the article text container closes."""
        if not self._inside_article_body:
            return

        if self._skip_depth > 0:
            self._skip_depth -= 1
        elif tag in self._block_tags:
            self._parts.append("\n")

        self._article_depth -= 1
        if self._article_depth <= 0:
            self._inside_article_body = False
            self._article_depth = 0

    def handle_data(self, data: str) -> None:
        """Collect visible article text outside skipped ad or related blocks."""
        if self._inside_article_body and self._skip_depth == 0:
            self._parts.append(data)

    def text(self) -> str:
        """Return clean paragraphs from the collected article body."""
        raw_text = html.unescape("".join(self._parts))
        lines = [re.sub(r"\s+", " ", line).strip() for line in raw_text.splitlines()]
        return "\n\n".join(line for line in lines if line)


def parse_scrape_date(value: date | str) -> date:
    """Parse a scrape date. The expected user format is DD-MM-YYYY."""
    if isinstance(value, date):
        return value

    for date_format in ("%d-%m-%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(value, date_format).date()
        except ValueError:
            pass

    raise ValueError("Use date format DD-MM-YYYY, for example '01-05-2025'.")


def parse_standard_timestamp(value: int | str) -> datetime:
    """Convert The Standard's Unix timestamp into Hong Kong time."""
    return datetime.fromtimestamp(int(value), tz=timezone.utc).astimezone(HK_TIMEZONE)


def safe_article_filename(value: str, max_length: int = 120) -> str:
    """Make an article title safe for a Windows filename while keeping it readable."""
    value = html.unescape(value).strip()
    value = re.sub(r'[<>:"/\\|?*]', "", value)
    value = re.sub(r"\s+", " ", value).strip(" .")
    return value[:max_length].strip(" .") or "Untitled article"


def remove_article_boilerplate(text: str) -> str:
    """Remove leftover clutter lines after article body extraction."""
    cleaned_lines: list[str] = []
    boilerplate_phrases = (
        "advertisement",
        "scroll to continue with content",
        "related articles",
        "more on this",
        "read more",
        "sign up for our newsletter",
        "subscribe to our newsletter",
        "follow us on",
        "share this article",
    )

    for line in text.splitlines():
        normalized = re.sub(r"\s+", " ", line).strip()
        if not normalized:
            continue

        lowered = normalized.lower()
        if lowered in {"share", "print", "email", "facebook", "x", "whatsapp", "telegram"}:
            continue
        if any(phrase in lowered for phrase in boilerplate_phrases):
            continue
        if re.match(r"^(photo|image|video|file photo):", lowered):
            continue

        cleaned_lines.append(normalized)

    return "\n\n".join(cleaned_lines)


def clean_standard_article_text(article_html: str) -> str:
    """Extract only the article content from a The Standard article page."""
    parser = StandardArticleTextExtractor()
    parser.feed(article_html or "")
    parser.close()
    return remove_article_boilerplate(parser.text())


def request_standard_content_path(
    content_path: str,
    session: requests.Session,
    timeout_seconds: int = 30,
) -> dict[str, Any]:
    """Request The Standard's client proxy for listing JSON.

    The normal page URL `/news?cursor=...` repeats the first rendered page. The website's
    infinite scroll uses `/api/content?path=/api/v1/cat/news/article?...`, so the scraper
    must use that proxy to reach older articles.
    """
    response = session.get(
        STANDARD_CONTENT_PROXY_URL,
        params={"path": content_path},
        timeout=timeout_seconds,
    )
    response.raise_for_status()
    return response.json()


def request_standard_articles_between_dates(
    start_date: date,
    end_date: date,
    session: requests.Session,
    max_pages_per_channel: int = 500,
    pause_seconds: float = 0.5,
    timeout_seconds: int = 30,
) -> list[dict[str, Any]]:
    """Download article metadata candidates from The Standard listing JSON."""
    articles_by_id: dict[str, dict[str, Any]] = {}

    for channel_slug in STANDARD_CHANNEL_SLUGS:
        next_path: str | None = f"/api/v1/cat/{channel_slug}/article"
        seen_paths: set[str] = set()

        for _ in range(max_pages_per_channel):
            if not next_path or next_path in seen_paths:
                break

            seen_paths.add(next_path)
            listing_data = request_standard_content_path(
                content_path=next_path,
                session=session,
                timeout_seconds=timeout_seconds,
            )

            page_articles = listing_data.get("data", [])
            for article in page_articles:
                published_hk = parse_standard_timestamp(article["publish_at"])
                if start_date <= published_hk.date() <= end_date:
                    articles_by_id[str(article["article_id"])] = article

            # The listing is newest first. After a page is entirely older than the start date, stop that channel.
            article_dates = [parse_standard_timestamp(article["publish_at"]).date() for article in page_articles]
            if article_dates and max(article_dates) < start_date:
                break

            next_path = listing_data.get("links", {}).get("next")
            sleep(pause_seconds)

    return sorted(articles_by_id.values(), key=lambda item: int(item["publish_at"]))


def scrape_standard_news(start_date: date | str, end_date: date | str) -> tuple[list[Path], Path]:
    """
    Scrape The Standard news articles for an inclusive date range.

    Parameters
    ----------
    start_date:
        First date to scrape. Use DD-MM-YYYY, for example '01-01-2025'.
    end_date:
        Last date to scrape. Use DD-MM-YYYY, for example '01-05-2025'.
        For one day, pass the same value for start_date and end_date.

    Returns
    -------
    tuple[list[Path], Path]
        First output: article text file paths saved in NEWS_DIR.
        Second output: one metadata CSV file path saved in METADATA_DIR.
    """
    # Validate the requested inclusive date range before doing any network work.
    start = parse_scrape_date(start_date)
    end = parse_scrape_date(end_date)
    if start > end:
        raise ValueError("start_date must be earlier than or equal to end_date.")

    # Create both output folders if this is the first run on the machine.
    NEWS_DIR.mkdir(parents=True, exist_ok=True)
    METADATA_DIR.mkdir(parents=True, exist_ok=True)

    session = requests.Session()
    session.headers.update(
        {
            # A browser-like user agent avoids blank/default clients being blocked by the site.
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) RAG-project-news-scraper/1.0",
            "Accept": "application/json,text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        }
    )

    article_records = request_standard_articles_between_dates(start_date=start, end_date=end, session=session)
    saved_news_files: list[Path] = []
    metadata_rows: list[dict[str, str]] = []
    filename_counts: dict[str, int] = {}

    for article in article_records:
        published_hk = parse_standard_timestamp(article["publish_at"])
        date_prefix = published_hk.strftime("%d-%m-%Y")
        article_title = html.unescape(str(article.get("title", "Untitled article")))
        article_url = urljoin(STANDARD_BASE_URL, str(article.get("url", "")))
        category = html.unescape(str(article.get("channel_name") or article.get("channel_slug_id") or ""))

        # Fetch the full article page because listing JSON only contains summaries.
        response = session.get(article_url, timeout=30)
        response.raise_for_status()
        article_text = clean_standard_article_text(response.text)

        # Skip empty pages instead of saving title/category boilerplate as article content.
        if not article_text.strip():
            print(f"Skipped empty article body: {article_url}")
            continue

        filename_stem = f"{date_prefix}_{safe_article_filename(article_title)}"
        filename_counts[filename_stem] = filename_counts.get(filename_stem, 0) + 1
        if filename_counts[filename_stem] > 1:
            filename_stem = f"{filename_stem}_{filename_counts[filename_stem]}"

        # Save only the cleaned article content. Do not prepend title, URL, date, or category.
        news_file = NEWS_DIR / f"{filename_stem}.txt"
        news_file.write_text(article_text.strip() + "\n", encoding="utf-8")
        saved_news_files.append(news_file)

        # Keep one CSV row per saved text file so metadata and files stay aligned.
        metadata_rows.append(
            {
                "articles": article_title,
                "released_date": date_prefix,
                "categories": category,
                "url": article_url,
            }
        )

        sleep(0.5)

    metadata_csv = METADATA_DIR / f"the_standard_metadata_{start.strftime('%d-%m-%Y')}_to_{end.strftime('%d-%m-%Y')}.csv"
    with metadata_csv.open("w", newline="", encoding="utf-8-sig") as csv_file:
        fieldnames = ["articles", "released_date", "categories", "url"]
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(metadata_rows)

    print(f"Saved {len(saved_news_files)} article text files to: {NEWS_DIR}")
    print(f"Saved metadata CSV to: {metadata_csv}")
    return saved_news_files, metadata_csv

## Run scraper

Use `DD-MM-YYYY` dates. The range is inclusive.

For one day, set `start_date` and `end_date` to the same value. The text files contain article body content only; title, URL, published time, and category are stored only in the metadata CSV.

In [2]:
# start_date, end_date = 'DD-MM-YYYY'
# Example for one day:
# news_files, metadata_csv = scrape_standard_news("01-05-2025", "01-05-2025")

# Example for a date range:
news_files, metadata_csv = scrape_standard_news("27-06-2026", "27-06-2026")

print(f"News text files: {len(news_files)}")
print(f"Metadata CSV: {metadata_csv}")

Saved 14 article text files to: C:\Program Files\Studying\coding\RAG_project\data\the_standard_news
Saved metadata CSV to: C:\Program Files\Studying\coding\RAG_project\data\the_standard_news_metadata\the_standard_metadata_27-06-2026_to_27-06-2026.csv
News text files: 14
Metadata CSV: C:\Program Files\Studying\coding\RAG_project\data\the_standard_news_metadata\the_standard_metadata_27-06-2026_to_27-06-2026.csv
